# Labelling defect candidates and training a classifier

The detector proposes candidates from fixed rules. This notebook lets you label
them by hand — **real / uncertain / artefact** — and trains a classifier on
those labels, so the pipeline can afterwards score new candidates by how much
they resemble the ones you kept.

The workflow, once per image:

1. detect candidates (histology or gel);
2. **click each marker** to cycle its label: blue (unset) → green (real) →
   amber (uncertain) → red (artefact);
3. save the labels.

Then, once several images are labelled, train and validate. Validation is
leave-one-image-out, so the score reflects an unseen image rather than a
memorised one.

One design choice worth knowing: **uncertain is not a training class.** With a
small hand-labelled set, splitting into three classes starves each one. Uncertain
rows are held out of training and only scored afterwards, so the model learns the
clean real-vs-artefact boundary and you still see where it lands on your doubtful
cases.

Requires a widget backend: run `%matplotlib widget` (Colab: `%matplotlib ipympl`
after `pip install ipympl`). The clicking will not work with the inline backend.

In [ ]:
#@title Setup { display-mode: "form" }
import os, subprocess, sys
REPO = "https://github.com/Danpc11/lung-nematic.git"  #@param {type:"string"}
ROOT = "/content/lung-nematic"
if not os.path.isdir(ROOT):
    subprocess.run(["git","clone","--depth","1",REPO,ROOT], check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","scikit-learn","ipympl"], check=True)
for n in [m for m in list(sys.modules) if m.startswith("lung_nematic")]:
    del sys.modules[n]
if ROOT not in sys.path: sys.path.insert(0, ROOT)

from lung_nematic.config import load_default_config
from lung_nematic.labeling import LabelingSession, load_labels
from lung_nematic.defect_features import extract_features
from lung_nematic.defect_classifier import train_classifier, grouped_cross_validate, CLASSES
import numpy as np, pandas as pd
print("ready. CLASSES =", CLASSES)

## 1 · Label one image

Edit the path, run, then click the markers. Run the save cell when done.

In [ ]:
#@title Detect candidates and open the labeller
%matplotlib widget
from dataclasses import replace
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

image_path = "data/histology/23-15_1.jpg"  #@param {type:"string"}
image_kind = "histology (collagen)"  #@param ["histology (collagen)", "phase-contrast gel"]

cfg = replace(load_default_config(), sigmas_px=(18.,25.,32.),
              density_quantile=0.30, min_scales_for_persistence=2,
              defect_grid_step_px=12, min_edge_distance_px=25)

rgb = np.array(Image.open(image_path).convert("RGB"))

if image_kind.startswith("phase"):
    from lung_nematic.phase_contrast import analyze_phase_contrast
    out = analyze_phase_contrast(rgb, cfg)
    field, candidates = out["field"], out["defects"]
else:
    from lung_nematic.preprocessing import make_tissue_mask
    from lung_nematic.collagen_field import detect_multiscale_collagen_defects
    mask, hed = make_tissue_mask(rgb)
    candidates, fields, _ = detect_multiscale_collagen_defects(hed[:,:,1], mask, cfg)
    rep = float(cfg.sigmas_px[len(cfg.sigmas_px)//2]); field = fields[rep]

image_id = image_path.split("/")[-1].rsplit(".",1)[0]
print(f"{len(candidates)} candidates in {image_id}")
session = LabelingSession(rgb, field, candidates, image_id=image_id)
session.show()

In [ ]:
#@title Save the labels for this image { display-mode: "form" }
label_dir = "labels"  #@param {type:"string"}
# A frame of pure noise? uncomment to mark every unlabelled candidate at once:
# session.label_all_unlabelled("artefact")
session.save(f"{label_dir}/{session.image_id}.csv")

Repeat cells 1–2 for every image. Aim for a spread of real and artefact examples; the classifier cannot learn a class it never sees.

In [ ]:
#@title 2 · Gather all labels and extract features { display-mode: "form" }
import glob
label_dir = "labels"  #@param {type:"string"}

labelled = load_labels(sorted(glob.glob(f"{label_dir}/*.csv")))
print(f"{len(labelled)} labelled candidates across "
      f"{labelled['image_id'].nunique()} images")
print("label mix:", labelled["label"].value_counts().to_dict())

# features are recomputed from the stored candidate geometry; the label CSVs
# carry x_px, y_px, charge and charge_raw, which is all extract_features needs
feature_rows = []
for image_id, group in labelled.groupby("image_id"):
    feats = extract_features(group, {"theta": np.zeros((2,2)),
                                     "order": np.zeros((2,2)),
                                     "density": np.ones((2,2))},
                             core_radius_px=25.0)
    # NOTE: for real use, re-extract with the actual field per image; the
    # labels CSV stores geometry, and the detector columns it kept
    feature_rows.append(feats.assign(image_id=image_id,
                                     label=group["label"].to_numpy()))
data = pd.concat(feature_rows, ignore_index=True)
print(f"feature matrix: {data.shape}")

In [ ]:
#@title 2 · Gather all labels and re-extract features from each image { display-mode: "form" }
#@markdown Features must come from the real director field, so each labelled
#@markdown image is re-analysed and its features matched to the saved labels by
#@markdown candidate position. Point `image_root` at where the images live.
import glob
from dataclasses import replace
from PIL import Image
Image.MAX_IMAGE_PIXELS = None

label_dir = "labels"  #@param {type:"string"}
image_root = "data"  #@param {type:"string"}
image_glob = "**/*.jpg"  #@param {type:"string"}

labelled = load_labels(sorted(glob.glob(f"{label_dir}/*.csv")))
print(f"{len(labelled)} labelled candidates across "
      f"{labelled['image_id'].nunique()} images")

cfg = replace(load_default_config(), sigmas_px=(18.,25.,32.),
              density_quantile=0.30, min_scales_for_persistence=2,
              defect_grid_step_px=12, min_edge_distance_px=25)

# index the available image files by their stem (= image_id)
image_files = {p.split("/")[-1].rsplit(".",1)[0]: p
               for p in glob.glob(f"{image_root}/{image_glob}", recursive=True)}

def field_for(image_id):
    """Re-run the detector on one image and return its representative field."""
    path = image_files[image_id]
    rgb = np.array(Image.open(path).convert("RGB"))
    from lung_nematic.preprocessing import make_tissue_mask
    from lung_nematic.collagen_field import compute_collagen_field
    mask, hed = make_tissue_mask(rgb)
    rep = float(cfg.sigmas_px[len(cfg.sigmas_px)//2])
    return compute_collagen_field(hed[:,:,1], rep, tissue_mask=mask,
                                  mask_normalized=cfg.mask_normalized_smoothing)

rows = []
for image_id, group in labelled.groupby("image_id"):
    if image_id not in image_files:
        print(f"  ! no image file found for {image_id}, skipping")
        continue
    field = field_for(image_id)
    feats = extract_features(group, field, core_radius_px=25.0)
    rows.append(feats.assign(image_id=image_id, label=group["label"].to_numpy()))

data = pd.concat(rows, ignore_index=True)
print(f"feature matrix from real fields: {data.shape}")
print("label mix:", data["label"].value_counts().to_dict())

## Reading the result

**The feature importances are the scientific output**, not the accuracy. If the
model leans on `scales_detected` and `order_core_annulus_ratio`, it has found
that the candidates you keep are the ones that persist across scales and have a
genuinely disordered core — an objective rule that now stands in for your eye.
If instead it leans on `x_px` or `nearest_neighbour_px`, be suspicious: it may
be memorising positions rather than learning defect quality.

**Leave-one-image-out is the honest number.** In-sample accuracy on this little
data will look excellent and mean nothing. The confusion matrix on held-out
images is what tells you whether the model generalises to an image it never saw.

**With only a handful of images, treat this as exploratory.** The classifier is
a way to make your labelling criteria explicit and testable, not yet a validated
filter. Its value grows with every labelled image.